In [1]:
import traceback, sys, csv, os
from datetime import datetime
from pathlib import Path
import pandas as pd
import numpy as np
from unidecode import unidecode

pd.set_option('display.max_columns', None)

src = Path.cwd().parent/"src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from routes.Paths import (
    Ruta_busquedad_contacto,
    Ruta_opl,
    Ruta_masterskill,
    Ruta_overall,
    Ruta_crm,
    Ruta_local,
    Ruta_global
)

from utils.Processing_OPL import data_opl
from utils.functions import load_files_from_folder
from utils.Processing_MasterSkill import limpiar_masterskill
from utils.Data_CRM.e_final_data_crm import transform_crm

require_columns = {
    "id de contacto",
    "canal",
    "marca de tiempo de inicio",
    "cola",
    "agente",
    "marca de tiempo de desconexion",
    "marca de tiempo de conexion a agente",
    "marca de tiempo en cola",
    "marca de tiempo de inicio de acw",
    "marca de tiempo de fin de acw"
}

delete_dup_columns = ["id de contacto"]

df_contactos = load_files_from_folder(
    Ruta_busquedad_contacto,
    require_columns,
    delete_dup_columns
)

require_columns_kpi = {
    "channel",
    "queue",
    "agent",
    "startinterval",
    "endinterval",
    "contacts answered in 20 seconds",
    "contacts answered in 180 seconds",
    "contacts handled",
    "contacts queued"
}
delete_dup_columns_kpi = [
    "channel",
        "queue",
        "agent",
        "startinterval"
]

df_kpi = load_files_from_folder(
    Ruta_overall,
    require_columns_kpi,
    delete_dup_columns_kpi
)

df_opl = data_opl(pd.read_excel(Ruta_opl, sheet_name="Combinado"))
df_opl["Correo_contratista"] = df_opl["Correo_contratista"].str.lower()
df_master = limpiar_masterskill(pd.read_excel(Ruta_masterskill, sheet_name="TB"))

In [2]:
df_kpi = df_kpi[df_kpi["channel"].notna() & df_kpi["queue"].notna()]

columnas_datetime = [
    "startinterval"
]

for col in columnas_datetime:
    df_kpi[col] = pd.to_datetime(df_kpi[col].astype(str)
                                 .str.rsplit("-", n=1)
                                 .str[0], errors="coerce")
    
df_kpi["fecha"] = pd.to_datetime(df_kpi["startinterval"].dt.date)
df_kpi["intervalo_inicio"] = df_kpi["startinterval"].dt.time
df_kpi["contactos_atendidos_en_umbral"] = np.where(
    (df_kpi["channel"].isin(["Voz", "Voice"])),
    df_kpi["contacts answered in 20 seconds"],
    df_kpi["contacts answered in 180 seconds"],
)
df_kpi["channel"] = np.where(
    df_kpi["channel"].isin(["Voice"]),
    "Voz",
    df_kpi["channel"]
)
df_kpi["queue"] = df_kpi["queue"].str.title().map(unidecode)
df_kpi["agent"] = df_kpi["agent"].str.lower()

In [3]:
df_kpi = df_kpi.merge(
    df_master[["IdMasterSkill", "Canal", "Colas"]],
    how="left",
    left_on=["channel", "queue"],
    right_on=["Canal", "Colas"]
).drop(columns=["Canal", "Colas", "startinterval", "endinterval"])

df_kpi = df_kpi.merge(
    df_opl[["Id_opl","Correo_contratista"]],
    how="left",
    left_on="agent",
    right_on="Correo_contratista",
).drop(columns=["Correo_contratista"])

In [ ]:
def guardar_data_kpioverall(
        df: pd.DataFrame,
        ruta_global: str,
        ruta_local: str,
        file_name: str = "07.Data_KpiOverAll.csv")-> None:

    df.to_csv(
        os.path.join(ruta_global, file_name),
        index=False,
        encoding="utf-8"
    )

    df.to_csv(
        os.path.join(ruta_local, file_name),
        index=False,
        encoding="utf-8"
    )

In [6]:
guardar_data_kpioverall(df_kpi, Ruta_global, Ruta_local)